In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder \
        .appName("Assignment_2") \
        .enableHiveSupport() \
        .getOrCreate()

26/02/24 02:05:40 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
movies = spark.read.format('csv').option('header',True).option("inferSchema","true").load('/tmp/spark_datasets/assignment_2/movies.csv')

movies.printSchema()

movies.show(10)

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
+-------+--------------------+--------------------+
only showing top 10 rows



In [3]:
# 1. Split the string into an array
movies = movies.withColumn('genres', split(col('genres'), '\|'))

# 2. Now that it's an array, you can explode it
movies_exploded = movies.withColumn("genres_array", explode(col("genres")))

movies_exploded.show(10)

+-------+--------------------+--------------------+------------+
|movieId|               title|              genres|genres_array|
+-------+--------------------+--------------------+------------+
|      1|    Toy Story (1995)|[Adventure, Anima...|   Adventure|
|      1|    Toy Story (1995)|[Adventure, Anima...|   Animation|
|      1|    Toy Story (1995)|[Adventure, Anima...|    Children|
|      1|    Toy Story (1995)|[Adventure, Anima...|      Comedy|
|      1|    Toy Story (1995)|[Adventure, Anima...|     Fantasy|
|      2|      Jumanji (1995)|[Adventure, Child...|   Adventure|
|      2|      Jumanji (1995)|[Adventure, Child...|    Children|
|      2|      Jumanji (1995)|[Adventure, Child...|     Fantasy|
|      3|Grumpier Old Men ...|   [Comedy, Romance]|      Comedy|
|      3|Grumpier Old Men ...|   [Comedy, Romance]|     Romance|
+-------+--------------------+--------------------+------------+
only showing top 10 rows



In [4]:
ratings = spark.read.format('csv').option('header',True).option("inferSchema","true").load('/tmp/spark_datasets/assignment_2/ratings.csv')

ratings.printSchema()

ratings.show(10)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|     70|   3.0|964982400|
|     1|    101|   5.0|964980868|
|     1|    110|   4.0|964982176|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
+------+-------+------+---------+
only showing top 10 rows



In [5]:
ratings = ratings.withColumn("timestamp", col("timestamp").cast(TimestampType()))

ratings.printSchema()

ratings.show(10)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      1|   4.0|2000-07-30 18:45:03|
|     1|      3|   4.0|2000-07-30 18:20:47|
|     1|      6|   4.0|2000-07-30 18:37:04|
|     1|     47|   5.0|2000-07-30 19:03:35|
|     1|     50|   5.0|2000-07-30 18:48:51|
|     1|     70|   3.0|2000-07-30 18:40:00|
|     1|    101|   5.0|2000-07-30 18:14:28|
|     1|    110|   4.0|2000-07-30 18:36:16|
|     1|    151|   5.0|2000-07-30 19:07:21|
|     1|    157|   5.0|2000-07-30 19:08:20|
+------+-------+------+-------------------+
only showing top 10 rows



In [6]:
tags = spark.read.format('csv').option('header',True).option("inferSchema","true").load('/tmp/spark_datasets/assignment_2/tags.csv')

tags.printSchema()

tags.show(10)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: integer (nullable = true)

+------+-------+-----------------+----------+
|userId|movieId|              tag| timestamp|
+------+-------+-----------------+----------+
|     2|  60756|            funny|1445714994|
|     2|  60756|  Highly quotable|1445714996|
|     2|  60756|     will ferrell|1445714992|
|     2|  89774|     Boxing story|1445715207|
|     2|  89774|              MMA|1445715200|
|     2|  89774|        Tom Hardy|1445715205|
|     2| 106782|            drugs|1445715054|
|     2| 106782|Leonardo DiCaprio|1445715051|
|     2| 106782|  Martin Scorsese|1445715056|
|     7|  48516|     way too long|1169687325|
+------+-------+-----------------+----------+
only showing top 10 rows



In [7]:
tags = tags.withColumn("timestamp", col("timestamp").cast(TimestampType()))

tags.printSchema()

tags.show(10)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)

+------+-------+-----------------+-------------------+
|userId|movieId|              tag|          timestamp|
+------+-------+-----------------+-------------------+
|     2|  60756|            funny|2015-10-24 19:29:54|
|     2|  60756|  Highly quotable|2015-10-24 19:29:56|
|     2|  60756|     will ferrell|2015-10-24 19:29:52|
|     2|  89774|     Boxing story|2015-10-24 19:33:27|
|     2|  89774|              MMA|2015-10-24 19:33:20|
|     2|  89774|        Tom Hardy|2015-10-24 19:33:25|
|     2| 106782|            drugs|2015-10-24 19:30:54|
|     2| 106782|Leonardo DiCaprio|2015-10-24 19:30:51|
|     2| 106782|  Martin Scorsese|2015-10-24 19:30:56|
|     7|  48516|     way too long|2007-01-25 01:08:45|
+------+-------+-----------------+-------------------+
only showing top 10 rows



# a. Show the aggregated number of ratings per year

In [8]:
ratings = ratings.withColumn('year',year(col('timestamp')))

agg_num_of_ratings_per_year = ratings.groupBy('year').agg(count('*').alias('total_ratings')).orderBy("year")

agg_num_of_ratings_per_year.show()

+----+-------------+
|year|total_ratings|
+----+-------------+
|1996|         6040|
|1997|         1916|
|1998|          507|
|1999|         2439|
|2000|        10061|
|2001|         3922|
|2002|         3478|
|2003|         4014|
|2004|         3279|
|2005|         5813|
|2006|         4059|
|2007|         7114|
|2008|         4351|
|2009|         4158|
|2010|         2301|
|2011|         1690|
|2012|         4656|
|2013|         1664|
|2014|         1439|
|2015|         6616|
+----+-------------+
only showing top 20 rows



In [9]:
agg_num_of_ratings_per_year_output_path='/tmp/spark_output/assignment_2/agg_num_of_ratings_per_year'

agg_num_of_ratings_per_year.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(agg_num_of_ratings_per_year_output_path)

# b. Show the average monthly number of ratings

In [10]:
ratings = ratings.withColumn('month',month(col('timestamp')))

average_of_ratings_per_month = ratings.groupBy('month').agg(avg(col('rating')).alias('average_ratings')).orderBy("month")

average_of_ratings_per_month.show()

+-----+------------------+
|month|   average_ratings|
+-----+------------------+
|    1|  3.50374251497006|
|    2|3.3519973804846104|
|    3| 3.455518018018018|
|    4| 3.613498123463181|
|    5| 3.450565101534503|
|    6| 3.417223796033994|
|    7| 3.639928057553957|
|    8|3.3590478289618693|
|    9|3.5998237367802584|
|   10|3.5126608841634024|
|   11| 3.634921455146755|
|   12|3.5149035651665694|
+-----+------------------+



In [11]:
average_of_ratings_per_month_output_path='/tmp/spark_output/assignment_2/average_of_ratings_per_month'

average_of_ratings_per_month.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(average_of_ratings_per_month_output_path)

# c. Show the rating levels distribution

In [12]:
rating_distribution = (ratings
                       .groupBy("rating")
                       .count()
                       .orderBy("rating"))

rating_distribution.show()

+------+-----+
|rating|count|
+------+-----+
|   0.5| 1370|
|   1.0| 2811|
|   1.5| 1791|
|   2.0| 7551|
|   2.5| 5550|
|   3.0|20047|
|   3.5|13136|
|   4.0|26818|
|   4.5| 8551|
|   5.0|13211|
+------+-----+



In [13]:
rating_distribution_output_path='/tmp/spark_output/assignment_2/rating_distribution'

rating_distribution.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(rating_distribution_output_path)

# d. Show the 18 movies that are tagged but not rated

In [14]:
tagged_movies = movies.join(tags, on="movieId", how="inner")

tagged_but_not_rated_movies = tagged_movies.join(ratings,on="movieId",how="left").where(col("rating").isNull()).select(col('title')).dropDuplicates().orderBy('movieId').limit(18)

tagged_but_not_rated_movies.show(truncate=False)

+--------------------------------------------+
|title                                       |
+--------------------------------------------+
|Innocents, The (1961)                       |
|Niagara (1953)                              |
|For All Mankind (1989)                      |
|Color of Paradise, The (Rang-e khoda) (1999)|
|I Know Where I'm Going! (1945)              |
|Chosen, The (1981)                          |
|Road Home, The (Wo de fu qin mu qin) (1999) |
|Scrooge (1970)                              |
|Proof (1991)                                |
|Parallax View, The (1974)                   |
|This Gun for Hire (1942)                    |
|Roaring Twenties, The (1939)                |
|Mutiny on the Bounty (1962)                 |
|In the Realms of the Unreal (2004)          |
|Twentieth Century (1934)                    |
|Call Northside 777 (1948)                   |
|Browning Version, The (1951)                |
|Chalet Girl (2011)                          |
+------------

In [15]:
tagged_but_not_rated_movies_output_path='/tmp/spark_output/assignment_2/tagged_but_not_rated_movies'

tagged_but_not_rated_movies.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(tagged_but_not_rated_movies_output_path)

# e. Show the movies that have rating but no tag

In [16]:
rated_movies = movies.join(ratings, on="movieId", how="inner")
rated_movies = rated_movies.withColumnRenamed('userId','user_Id')
rated_but_not_tagged_movies = rated_movies.join(tags,on="movieId",how="left").where(col("tag").isNull()).drop('tag','timestamp','userId').orderBy('movieId')

rated_but_not_tagged_movies.show(truncate=False)

+-------+------------------------+-------------------------+-------+------+----+-----+
|movieId|title                   |genres                   |user_Id|rating|year|month|
+-------+------------------------+-------------------------+-------+------+----+-----+
|4      |Waiting to Exhale (1995)|[Comedy, Drama, Romance] |14     |3.0   |1996|6    |
|4      |Waiting to Exhale (1995)|[Comedy, Drama, Romance] |6      |3.0   |1996|10   |
|4      |Waiting to Exhale (1995)|[Comedy, Drama, Romance] |84     |3.0   |1997|3    |
|4      |Waiting to Exhale (1995)|[Comedy, Drama, Romance] |162    |3.0   |1996|7    |
|4      |Waiting to Exhale (1995)|[Comedy, Drama, Romance] |262    |1.0   |1996|8    |
|4      |Waiting to Exhale (1995)|[Comedy, Drama, Romance] |411    |2.0   |1996|6    |
|4      |Waiting to Exhale (1995)|[Comedy, Drama, Romance] |600    |1.5   |2009|3    |
|6      |Heat (1995)             |[Action, Crime, Thriller]|84     |4.0   |1997|3    |
|6      |Heat (1995)             |[Action, 

In [17]:
output_df = rated_but_not_tagged_movies.withColumn("genres", concat_ws("|", col("genres")))

rated_but_not_tagged_movies_output_path='/tmp/spark_output/assignment_2/rated_but_not_tagged_movies'

output_df.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(rated_but_not_tagged_movies_output_path)

# f. Focusing on the rated untagged movies with more than 30 user ratings, show the top 10 movies in terms of average rating and number of ratings

In [18]:
rated_but_not_tagged_movies_with_more_than_30_users = rated_but_not_tagged_movies.groupBy('movieId','title').agg(count('*').alias('total_users')).filter(col('total_users') > 30).orderBy(col('total_users').desc()).limit(10)

rated_but_not_tagged_movies_with_more_than_30_users.show(truncate=False)

+-------+--------------------------------------------+-----------+
|movieId|title                                       |total_users|
+-------+--------------------------------------------+-----------+
|2858   |American Beauty (1999)                      |204        |
|344    |Ace Ventura: Pet Detective (1994)           |161        |
|367    |Mask, The (1994)                            |157        |
|1036   |Die Hard (1988)                             |145        |
|165    |Die Hard: With a Vengeance (1995)           |144        |
|1265   |Groundhog Day (1993)                        |143        |
|231    |Dumb & Dumber (Dumb and Dumber) (1994)      |133        |
|10     |GoldenEye (1995)                            |132        |
|4886   |Monsters, Inc. (2001)                       |132        |
|2683   |Austin Powers: The Spy Who Shagged Me (1999)|121        |
+-------+--------------------------------------------+-----------+



In [19]:
rated_but_not_tagged_movies_with_more_than_30_users_output_path='/tmp/spark_output/assignment_2/rated_but_not_tagged_movies_with_more_than_30_users'

rated_but_not_tagged_movies_with_more_than_30_users.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(rated_but_not_tagged_movies_with_more_than_30_users_output_path)

# g. What is the average number of tags per movie in tagsDF? 

In [20]:
average_tags_per_movie = tags.groupBy("movieId").agg(count("*").alias("tag_count"))

average_numbers_of_tags = average_tags_per_movie.select(round(avg("tag_count"),2).alias('average_tag_count_per_movie'))

print(f"The average number of tags per movie is:")
average_numbers_of_tags.show()

The average number of tags per movie is:
+---------------------------+
|average_tag_count_per_movie|
+---------------------------+
|                       2.34|
+---------------------------+



# And the average number of tags per user?

In [21]:
average_tags_per_user = tags.groupBy(col('userId')).agg(count("*").alias("tag_count"))

average_tags_per_user = average_tags_per_user.select(round(avg("tag_count"),2).alias('average_tag_count_per_user'))

print(f"The average number of tags per user is:")
average_tags_per_user.show()

The average number of tags per user is:
+--------------------------+
|average_tag_count_per_user|
+--------------------------+
|                      63.5|
+--------------------------+



#  How does it compare with the average number of tags a user assigns to a movie?

In [22]:
tags_per_user_movie = tags.groupBy("userId", "movieId").agg(count("*").alias("tags_given"))


avg_tags_per_habit = tags_per_user_movie.select(round(avg("tags_given"), 2).alias("avg_tags_per_user_movie"))

print("Average tags a user assigns to a single movie:")
avg_tags_per_habit.show()

Average tags a user assigns to a single movie:
+-----------------------+
|avg_tags_per_user_movie|
+-----------------------+
|                   2.07|
+-----------------------+



In [23]:
df1 = average_numbers_of_tags.withColumn("metric", lit("Average Tags per Movie")) \
                             .select(col("metric"), col("average_tag_count_per_movie").alias("value"))

df2 = average_tags_per_user.withColumn("metric", lit("Average Tags per User")) \
                           .select(col("metric"), col("average_tag_count_per_user").alias("value"))

df3 = avg_tags_per_habit.withColumn("metric", lit("Avg Tags per User per Movie")) \
                         .select(col("metric"), col("avg_tags_per_user_movie").alias("value"))

# Union them into one table
summary_df = df1.union(df2).union(df3)

tag_metrics_summary_output_path='/tmp/spark_output/assignment_2/tag_metrics_summary'


# Store in one CSV
summary_df.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(tag_metrics_summary_output_path)

summary_df.show(truncate=False)

+---------------------------+-----+
|metric                     |value|
+---------------------------+-----+
|Average Tags per Movie     |2.34 |
|Average Tags per User      |63.5 |
|Avg Tags per User per Movie|2.07 |
+---------------------------+-----+



# h. Identify the users that tagged movies without rating them

In [24]:
users_who_rated = ratings.select("userId").distinct()

users_tagging_only = tags.select("userId").distinct().join(users_who_rated, on="userId", how="left_anti")

print("Users who tagged movies but never rated any:")
users_tagging_only.show()

Users who tagged movies but never rated any:
+------+
|userId|
+------+
+------+



In [25]:
users_tagging_only_output_path='/tmp/spark_output/assignment_2/users_tagging_only'

users_tagging_only.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(users_tagging_only_output_path)

# i. What is the average number of ratings per user in ratings DF? 

In [10]:
count_rating_per_user = ratings.groupBy('userId').count()

average_rating_per_user = count_rating_per_user.agg(avg('count').alias('average_rating_per_user'))

average_rating_per_user.show()

+-----------------------+
|average_rating_per_user|
+-----------------------+
|     165.30491803278687|
+-----------------------+



# And the average number of ratings per movie?

In [11]:
count_rating_per_movie = ratings.groupBy('movieId').count()

average_rating_per_movie = count_rating_per_movie.agg(avg('count').alias('average_rating_per_movie'))

average_rating_per_movie.show()

+------------------------+
|average_rating_per_movie|
+------------------------+
|      10.369806663924312|
+------------------------+



In [12]:
df1 = average_rating_per_user.withColumn('metric',lit(' Average Rating Per User'))\
                             .select(col("metric"), col("average_rating_per_user").alias("value"))

df2 = average_rating_per_movie.withColumn('metric',lit('Average Rating Per Movie'))\
                             .select(col("metric"), col("average_rating_per_movie").alias("value"))
summary_df = df1.union(df2)

summary_df.show()

average_rating_per_user_per_movie_output_path='/tmp/spark_output/assignment_2/average_rating_per_user_per_movie'

summary_df.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(average_rating_per_user_per_movie_output_path)

+--------------------+------------------+
|              metric|             value|
+--------------------+------------------+
| Average Rating P...|165.30491803278687|
|Average Rating Pe...|10.369806663924312|
+--------------------+------------------+



# j. What is the predominant (frequency based) genre per rating level?

In [13]:
from pyspark.sql.window import Window
freq_of_genre_per_rating = ratings.join(movies_exploded,on="movieId")

freq_of_genre_per_rating = freq_of_genre_per_rating.groupBy("rating", "genres_array").agg(count('*').alias('genre_freq'))

window_spec = Window.partitionBy("rating").orderBy(desc("genre_freq"))

predominant_genres = (freq_of_genre_per_rating
                      .withColumn("rank", row_number().over(window_spec))
                      .filter(col("rank") == 1)
                      .select("rating", col("genres_array").alias("predominant_genre"), "genre_freq")
                      .orderBy("rating"))

predominant_genres.show()

+------+-----------------+----------+
|rating|predominant_genre|genre_freq|
+------+-----------------+----------+
|   0.5|           Comedy|       632|
|   1.0|           Comedy|      1317|
|   1.5|           Comedy|       895|
|   2.0|           Comedy|      3405|
|   2.5|           Comedy|      2530|
|   3.0|           Comedy|      8306|
|   3.5|            Drama|      5514|
|   4.0|            Drama|     12360|
|   4.5|            Drama|      4217|
|   5.0|            Drama|      6350|
+------+-----------------+----------+



In [14]:
predominant_genres_output_path='/tmp/spark_output/assignment_2/predominant_genres'

predominant_genres.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(predominant_genres_output_path)

# k. What is the predominant tag per genre and the most tagged genres?

In [15]:
from pyspark.sql.window import Window
freq_of_tag_per_rating = tags.join(movies_exploded,on="movieId")

freq_of_tag_per_rating = freq_of_tag_per_rating.groupBy("tag", "genres_array").agg(count('*').alias('tag_freq'))

window_spec = Window.partitionBy("tag").orderBy(desc("tag_freq"))

predominant_tags = (freq_of_tag_per_rating
                      .withColumn("rank", row_number().over(window_spec))
                      .filter(col("rank") == 1)
                      .select("tag", col("genres_array").alias("predominant_tag"), "tag_freq")
                      .orderBy(col("tag_freq").desc()))

predominant_tags.show()

+-----------------+---------------+--------+
|              tag|predominant_tag|tag_freq|
+-----------------+---------------+--------+
| In Netflix queue|          Drama|      93|
|      atmospheric|          Drama|      28|
|           Disney|       Children|      23|
|        superhero|         Action|      23|
|           sci-fi|         Sci-Fi|      21|
|thought-provoking|          Drama|      21|
|            funny|         Comedy|      18|
|      dark comedy|         Comedy|      17|
|         religion|          Drama|      17|
|          surreal|          Drama|      17|
|         suspense|       Thriller|      17|
|      time travel|         Sci-Fi|      16|
|     twist ending|       Thriller|      16|
|           aliens|         Sci-Fi|      15|
|       psychology|          Drama|      15|
|           comedy|         Comedy|      14|
|   mental illness|          Drama|      14|
|           quirky|         Comedy|      14|
|            crime|          Crime|      13|
|         

In [16]:
predominant_tags_output_path='/tmp/spark_output/assignment_2/predominant_tags'

predominant_tags.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(predominant_tags_output_path)

# l. What are the most predominant (popularity based) movies?

In [17]:
movie_popularity = ratings.groupBy("movieId").agg(count("*").alias("rating_count"))

popular_movies = movie_popularity.join(movies, on="movieId", how="inner").select("movieId", "title", "rating_count").orderBy(desc("rating_count"))

print("Top 10 Most Popular Movies (By Rating Count):")
popular_movies.show(10, truncate=False)

Top 10 Most Popular Movies (By Rating Count):
+-------+-----------------------------------------+------------+
|movieId|title                                    |rating_count|
+-------+-----------------------------------------+------------+
|356    |Forrest Gump (1994)                      |329         |
|318    |Shawshank Redemption, The (1994)         |317         |
|296    |Pulp Fiction (1994)                      |307         |
|593    |Silence of the Lambs, The (1991)         |279         |
|2571   |Matrix, The (1999)                       |278         |
|260    |Star Wars: Episode IV - A New Hope (1977)|251         |
|480    |Jurassic Park (1993)                     |238         |
|110    |Braveheart (1995)                        |237         |
|589    |Terminator 2: Judgment Day (1991)        |224         |
|527    |Schindler's List (1993)                  |220         |
+-------+-----------------------------------------+------------+
only showing top 10 rows



In [18]:
popular_movies_output_path='/tmp/spark_output/assignment_2/popular_movies'

popular_movies.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(popular_movies_output_path)

# m. Top 10 movies in terms of average rating (provided more than 30 users reviewed them)

In [19]:
movies_by_avg_ratings = ratings.groupBy("movieId").agg(avg(col('rating')).alias("average_rating")).orderBy(col("average_rating").desc())

movies_by_user_ratings = ratings.groupBy("movieId").agg(count('*').alias("total_rated_users")).filter(col("total_rated_users") > 30).orderBy(col("total_rated_users").desc())

top_10_movies_by_avg_rating_and_more_than_30_users = movies_by_avg_ratings.join(movies_by_user_ratings,on="movieId",how="inner").orderBy(col("average_rating").desc(),col("total_rated_users").desc()).limit(10)

top_10_movies_by_avg_rating_and_more_than_30_users.show()

+-------+-----------------+-----------------+
|movieId|   average_rating|total_rated_users|
+-------+-----------------+-----------------+
|    318|4.429022082018927|              317|
|   1204|              4.3|               45|
|    858|        4.2890625|              192|
|   2959|4.272935779816514|              218|
|   1276|4.271929824561403|               57|
|    750|4.268041237113402|               97|
|    904|4.261904761904762|               84|
|   1221| 4.25968992248062|              129|
|  48516|4.252336448598131|              107|
|   1213|             4.25|              126|
+-------+-----------------+-----------------+



In [20]:
top_10_movies_by_avg_rating_and_more_than_30_users_output_path='/tmp/spark_output/assignment_2/top_10_movies_by_avg_rating_and_more_than_30_users'

top_10_movies_by_avg_rating_and_more_than_30_users.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(top_10_movies_by_avg_rating_and_more_than_30_users_output_path)

In [21]:
spark.stop()